In [3]:
import os
import tensorflow as tf
import json
from transformers import GPT2Tokenizer , TFGPT2LMHeadModel
import requests

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.padding_side="left"
model = TFGPT2LMHeadModel.from_pretrained("gpt2")

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Import Llama3

In [5]:
def generate_response_llama3(prompt,max_length=150):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
    "model":"llama3",
    "prompt":prompt,
    "stream": False,
    "temperature": 0.1,
    "options": {
        "num_predict":max_length
    }
        }
    )
    return response.json()["response"]

In [6]:
def generate_response_gpt2(input):
    #tokenized_input = tokenizer(input,return_tensors="tf",truncation=True)
    #prompt  = build_prompt(input)
    input_ids = tokenizer.encode(input,
                                return_tensors='tf',
                                max_length=512,
                                truncation=True)
    output_ids = model.generate(
        input_ids,
        num_beams=4,
        repetition_penalty=1.5,
        pad_token_id = tokenizer.eos_token_id,
        eos_token_id = tokenizer.eos_token_id,
        do_sample=False,
        #max_length=input_ids.shape[1] + 80,
        early_stopping=True,
        max_new_tokens=50
    )

    output = tokenizer.decode(output_ids[0],skip_special_tokens=True)
    response = output[len(input):].strip()

    # stop if model starts another User/Bot turn
    for stop_word in ["User:", "Bot:"]:
        if stop_word in response:
            response = response.split(stop_word)[0].strip()
    return response

In [7]:
def Prompt_generator(goal):
    modified_goal = goal.lower().strip()

    if "exam" in modified_goal :
        return [
            "Identify the Exam Topic",
            "Divide the topics into study sessions",
            "Review important concepts",
            "Practice with exercises or old exams",
            "Do a final revision"
        ]
    elif "project" in modified_goal :
        return [
            "Define the project goal",
            "Break the project into modules",
            "Implement one module at a time",
            "Test the implementation",
            "Improve the final result"
        ]
    elif "schedule" in modified_goal or "week" in modified_goal:
        return [
            "List the important tasks for the week",
            "Set priorities for each task",
            "Distribute tasks across days",
            "Reserve time for breaks and review"
        ]
    else :
        return [
            "Understand the goal",
            "Break the goal into small tasks",
            "Set priorities",
            "Start with the important task"
        ]

In [8]:
class Memory:
    def __init__(self):
        self.history=[]
        self.goal=""
    def add(self,role,content):
        self.history.append(f"{role}: {content}")
        self.history = self.history[-4:]
    def delete(self):
        self.history = []
        self.goal = []
    def setGoal(self,goal):
        self.goal = goal
    def getGoals(self):
        return self.goal
    def getHistoryText(self):
        return "\n".join(self.history)

In [19]:
'''class TaskAgent:
    def run(self, user_input):
        steps = Prompt_generator(user_input)
        steps_text = "\n".join(steps)
        full_prompt = f"Goal: {user_input}\nSteps:\n{steps_text}\nUser:Follow the Instructions and give me response\nBot:"
        response = generate_response_llama3(full_prompt)
        return response'''
class TaskAgent:
    def run(self, user_input, desc_agent=None, est_agent=None):
        steps = Prompt_generator(user_input)
        result = [f"📋 Plan for: {user_input}\n"]

        for i, step in enumerate(steps, 1):
            line = f"📌 Step {i}: {step}"

            if desc_agent:                             
                line += f"\n    {desc_agent.describe_step(step)}"

            if est_agent:                              
                line += f"\n   ⏱ {est_agent.estimate_step(step)}"

            result.append(line)

        return "\n\n".join(result)

In [22]:
class DescriptiveAgent:
    def run(self, steps):
        prompt = f"""
You are a Descriptive Agent.

Explain clearly how to execute these steps:

{steps}
"""
        return generate_response_llama3(prompt)

In [21]:
class EstimationAgent:
    def run(self, steps):
        prompt = f"""
You are an Estimation Agent.

Estimate the time required for each of the following steps.

Steps:
{steps}

Give a realistic estimation for each step and total time.
"""
        return generate_response_llama3(prompt)

In [25]:
class CoordinatorAgent:
    def __init__(self):
        self.task_agent = TaskAgent()
        self.estimation_agent = EstimationAgent()
        self.descriptive_agent = DescriptiveAgent()
        self.history_agent = Memory()

    def route(self, user_input):
        u = user_input.lower()

        if "plan" in u or "task" in u:
            steps = self.task_agent.run(user_input)
            return f"PLAN:\n{steps}"

        elif "estimate" in u:
            steps = self.task_agent.run(user_input)
            estimation = self.estimation_agent.run(steps)
            return f"STEPS:\n{steps}\n\nTIME ESTIMATION:\n{estimation}"

        elif "describe" in u:
            steps = self.task_agent.run(user_input)
            description = self.descriptive_agent.run(steps)
            return f"STEPS:\n{steps}\n\nDESCRIPTION:\n{description}"

        elif "full" in u:
            steps = self.task_agent.run(user_input)
            estimation = self.estimation_agent.run(steps)
            description = self.descriptive_agent.run(steps)

            return f"""
PLAN:
{steps}

TIME ESTIMATION:
{estimation}

DESCRIPTION:
{description}
"""

        else:
            return "I didn't understand the request."

In [26]:
coordination = CoordinatorAgent()
while True:
  user_input = input("user :")
  if user_input == "exit":
    break
  response = coordination.route(user_input)
  coordination.history_agent.add("user:",user_input)
  coordination.history_agent.add("Bot:",response)
  if user_input not in coordination.history_agent.getGoals():
    coordination.history_agent.setGoal(user_input)
  print("\n================ RESULT ================\n")
  print(response)

user : hello



================ RESULT ================

I didn't understand the request.


user : full build a web platform for students



================ RESULT ================


PLAN:
📋 Plan for: full build a web platform for students


📌 Step 1: Understand the goal

📌 Step 2: Break the goal into small tasks

📌 Step 3: Set priorities

📌 Step 4: Start with the important task

TIME ESTIMATION:
As an Estimation Agent, I'll provide you with a realistic estimate of the time required for each step. Here are my estimates:

**Step 1: Understand the goal**

Estimated time: 2-4 hours (depending on the complexity of the project)

This step involves clarifying the objective of building a web platform for students, understanding the requirements, and defining the scope of the project. This may involve discussions with stakeholders, reviewing existing documentation, or conducting research.

**Step 2: Break the goal into small tasks**

Estimated time: 4-8 hours (depending on the level of detail required)

Breaking down the overall goal into smaller, manageable tasks is essential for successful project planning. This step requires i

user : how to do a homework


...

user : full how to do a homework



================ RESULT ================


PLAN:
📋 Plan for: full how to do a homework


📌 Step 1: Understand the goal

📌 Step 2: Break the goal into small tasks

📌 Step 3: Set priorities

📌 Step 4: Start with the important task

TIME ESTIMATION:
As an Estimation Agent, I'll provide a realistic estimate of the time required for each step. Here's my assessment:

**Step 📋: Plan for: full how to do a homework**

* Time estimate: 5-10 minutes

This initial planning phase will set the stage for the entire process. It's essential to clarify what needs to be done and allocate sufficient time for each task.

**Step 📌 Step 1: Understand the goal**

* Time estimate: 2-5 minutes

Understanding the homework goal is crucial to ensure you're on the right track. Take a few minutes to review the assignment, ask clarifying questions if needed, and confirm what needs to be done.

**Step 📌 Step

DESCRIPTION:
As your Descriptive Agent, I'll guide you through a step-by-step process to help you execute the

user : estimate for me the time needed to be rich 



================ RESULT ================

STEPS:
📋 Plan for: estimate for me the time needed to be rich 


📌 Step 1: Understand the goal

📌 Step 2: Break the goal into small tasks

📌 Step 3: Set priorities

📌 Step 4: Start with the important task

TIME ESTIMATION:
As an Estimation Agent, I'll provide you with a realistic estimate of the time required for each step to achieve the goal "estimate for me the time needed to be rich". Here are my estimates:

📌 Step 1: Understand the goal (Estimated time: 30 minutes)

* Understanding what being "rich" means to you, and defining your financial goals.
* Clarifying any misconceptions or assumptions about wealth.

Total estimated time: 0.5 hours

📌 Step 2: Break the goal into small tasks (Estimated time: 1 hour)

* Identifying specific areas of focus for achieving financial success:
	+ Investing
	+ Saving
	+ Increasing income
	+ Reducing expenses
*


user : exit
